In [ ]:
import os
os.environ.pop("ALL_PROXY", None)
os.environ.pop("all_proxy", None)

In [ ]:
pip install open_clip_torch

In [ ]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [ ]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [ ]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

# Zero Shot on ImageNet-V2 dataset

### Load the dataset

In [ ]:
pip install git+https://github.com/modestyachts/ImageNetV2_pytorch

In [ ]:
from imagenetv2_pytorch import ImageNetV2Dataset

dataset = ImageNetV2Dataset("matched-frequency", location= "../data", transform=preprocess) # supports matched-frequency, threshold-0.7, top-images variants
dataloader = DataLoader(dataset, batch_size=32, num_workers = 2) # use whatever batch size you wish

In [ ]:
for batch_idx, (image, label) in enumerate(dataloader):
    print(f"Batch ID: {batch_idx} Image Shape: {image.shape}, Label Shape: {label.shape}")
    break

### Build the text features

In [ ]:
from imagenet_classes import IMAGENET_CLASS_NAMES, IMAGENET_TEMPLATES

In [ ]:
print(f"{len(IMAGENET_CLASS_NAMES)}, {len(IMAGENET_TEMPLATES)}")

In [ ]:
text_features = build_and_cache_text_features(model, tokenizer, IMAGENET_CLASS_NAMES, IMAGENET_TEMPLATES, device, "../features", "imagenet1k_text_features")

### Build the Image Features

In [ ]:
image_features_and_labels = build_and_cache_image_features(model, device, dataloader, "../features", "imagenetv2")

### Zero Shot Evaluation

Top-1 accuracy: 53.21    
Top-5 accuracy: 79.50

In [ ]:
image_features = image_features_and_labels['image_features'].to(device)
labels = image_features_and_labels['labels'].to(device)

similarity = image_features @ text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, labels, topk=(1, 5))

In [ ]:
n = len(labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")

### Test the cached features

In [ ]:
features = load_cached_features("../features/imagenetv2.pt", "../features/imagenet1k_text_features.pt")

In [ ]:
image_features = features["image_features"]
labels = features["labels"]
text_features = features["text_features"]

similarity = image_features @ text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, labels, topk=(1, 5))

In [ ]:
n = len(labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")

# Zero Shot on ImageNet-R Dataset

### Download the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset("axiong/imagenet-r")

In [ ]:
print(ds)
print(ds['test'].features)
print(ds["test"][0])
print(len(set(ds["test"]["class_name"])))

## Prepare Class Names and Labels

In [ ]:
unique_pairs = sorted(set(zip(ds['test']["wnid"], ds['test']["class_name"])))
r_wnids = [pair[0] for pair in unique_pairs]
r_class_names = [pair[1] for pair in unique_pairs]

print(len(r_wnids))
print(len(r_wnids))

In [ ]:
wnid_to_r_index = {wnid: i for i, wnid in enumerate(r_wnids)}
print(wnid_to_r_index)

### Prepare the Text Features

In [ ]:
r_text_features = build_and_cache_text_features(model, tokenizer, r_class_names, IMAGENET_TEMPLATES, device, "../data", "imagenet_r_text_features")

### Load The Dataset

In [ ]:
class ImageNetRDataset(Dataset):
    def __init__(self, dataset, preprocess, wnid_to_index):
        self.dataset = dataset
        self.preprocess = preprocess
        self.wnid_to_index = wnid_to_index

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        example = self.dataset[idx]
        image = preprocess((example["image"]))
        label = wnid_to_r_index[example["wnid"]]
        return image, label

In [ ]:
r_dataset = ImageNetRDataset(ds['test'], preprocess, wnid_to_r_index)
r_dataloader = DataLoader(r_dataset, batch_size=32, num_workers=2)

In [ ]:
images, labels = next(iter(r_dataloader))
print(images.shape, labels.shape)
print(labels[:5])

### Prepare the Image Features

In [ ]:
r_image_cach = build_and_cache_image_features(model, device, r_dataloader, "../data", "imagenet-r")

### Evaluation

Top-1 accuracy: 72.80  
Top-5 accuracy: 90.51

In [ ]:
r_image_features = r_image_cach['image_features'].to(device)
r_labels = r_image_cach['labels'].to(device)
r_text_features = r_text_features.to(device)

similarity = r_image_features @ r_text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, r_labels, topk=(1, 5))

In [ ]:
n = len(labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")

# Zero Shot on ImageNet-Sketch Dataset

### Download and extract the dataset

In [ ]:
!gdown --id 1Mj0i5HBthqH1p_yeXzsg22gZduvgoNeA

In [ ]:
!unzip -qq ImageNet-Sketch.zip

### Load the dataset

In [ ]:
import torchvision.datasets as datasets

sk_dataset = datasets.ImageFolder(root = "./sketch", transform=preprocess)
sk_dataloader = DataLoader(sk_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
import os
folders = sorted(os.listdir("./sketch"))
print(len(folders))
print(folders[:10])

In [ ]:
print(sk_dataset.classes[:5])

### Prepare the classes and labels
The mapping was identical to imagenetV2. so this section is not necessary.

In [ ]:
# import json
# from torchvision.datasets.utils import download_url

# # Download the official ImageNet class index mapping
# download_url(
#     "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json",
#     "../data",
#     "imagenet_class_index.json",
# )

# # Load the JSON mapping file
# with open("../data/imagenet_class_index.json", "r") as f:
#     class_idx = json.load(f)

# # Convert to a list where index 0-999 corresponds to model outputs
# class_names = [class_idx[str(i)][1] for i in range(1000)]

# # Example: Print class name for index 263
# print(class_names[263])  # Output: Pembroke (welsh_corgi)

In [ ]:
# wnid_to_index = {val[0]: int(key) for key, val in class_idx.items()}

# imagefolder_to_original = {
#     i: wnid_to_index[wnid] for i, wnid in enumerate(sk_dataset.classes)
# }

# is_identity = all(imagefolder_to_original[i] == i for i in range(1000))
# print(is_identity)

### Build the text features
-- The mappings are identical to imagenet-V2 dataset. So, precoumputed text features will be used here.

In [ ]:
from clip_zeroshot import load_cached_text_features

sk_text_features = load_cached_text_features('/kaggle/working/imagenet1k_text_features.pt')

### Build image features

In [ ]:
sk_img_cached = build_and_cache_image_features(model, device, sk_dataloader, '.', 'sketch_image_features')

### Zero Shot Evaluation

Top-1 accuracy: 44.24  
Top-5 accuracy: 72.10

In [ ]:
sk_image_features = cached_img['image_features'].to(device)
sk_labels = cached_img['labels'].to(device)
sk_text_features = sk_text_features['text_features']
sk_text_features = sk_text_features.to(device)

similarity = sk_image_features @ sk_text_features
logits = 100 * similarity
acc1, acc5 = top_k_accuracy(logits, sk_labels, topk=(1, 5))

In [ ]:
n = len(sk_labels)

top1 = (acc1 / n) * 100
top5 = (acc5 / n) * 100

print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")

# Zero Shot on DG benchmark PACS

### Load the datasets

In [ ]:
domains = ["photo", "art_painting", "cartoon", "sketch"]

pacs_datasets = {
    domain: datasets.ImageFolder(root=f"/kaggle/input/datasets/nickfratto/pacs-dataset/pacs_data/pacs_data/{domain}", transform=preprocess)
    for domain in domains
}

pacs_dataloaders = {
    domain: DataLoader(pacs_datasets[domain], batch_size=32, shuffle=False, num_workers=2)
    for domain in domains
}

### Inspect the class names for consistency

In [ ]:
for domain in domains:
    print(domain, pacs_datasets[domain].classes)

In [ ]:
for domain in domains:
    print(domain, pacs_datasets[domain].class_to_idx)

In [ ]:
mappings = [pacs_datasets[d].class_to_idx for d in domains]
all_match = all(m == mappings[0] for m in mappings)
print(all_match)

### Build the text features

In [ ]:
pacs_classes = [cls for cls in pacs_datasets['photo'].classes]
print(pacs_classes)

In [ ]:
pacs_text_features = build_and_cache_text_features(model, tokenizer, pacs_classes, IMAGENET_TEMPLATES, device, './', 'pacs_text_features')

### Build the image features and evalate

Top-1 accuracy for domain photo: 99.94  
Top-5 accuracy for domain photo: 100.00

Top-1 accuracy for domain art_painting: 96.73  
Top-5 accuracy for domain art_painting: 100.00

Top-1 accuracy for domain cartoon: 98.72  
Top-5 accuracy for domain cartoon: 100.00

Top-1 accuracy for domain sketch: 87.99  
Top-5 accuracy for domain sketch: 99.95

In [ ]:
for domain in domains:
    cached = build_and_cache_image_features(model, device, pacs_dataloaders[domain], "./", f"pacs_{domain}")

    pacs_image_features = cached['image_features'].to(device)
    pacs_labels = cached['labels'].to(device)
    pacs_text_features = pacs_text_features.to(device)

    similarity = pacs_image_features @ pacs_text_features
    logits = 100 * similarity
    acc1, acc5 = top_k_accuracy(logits, pacs_labels, topk=(1, 5))

    n = len(pacs_labels)

    top1 = (acc1 / n) * 100
    top5 = (acc5 / n) * 100

    print(f"Top-1 accuracy for domain {domain}: {top1:.2f}")
    print(f"Top-5 accuracy for domain {domain}: {top5:.2f}")